# DeepPurpose GNN

Graph neural network model for LD50 regression from SMILES.

In [10]:
from pathlib import Path
import json
import sys

import numpy as np
from DeepPurpose import CompoundPred, utils

PROJECT_ROOT = Path('../..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from utils.modeling import TARGET_COLUMN, SMILES_COLUMN, artifact_paths, load_smiles_splits, plot_pred_vs_real, regression_metrics, save_predictions


In [11]:
RUN_ID = 'deeppurpose_gnn'
DRUG_ENCODING = 'DGL_GCN'
splits = load_smiles_splits()
paths = artifact_paths(Path.cwd(), RUN_ID, '')


In [12]:
train_data = utils.data_process(X_drug=splits['train'][SMILES_COLUMN].values, y=splits['train'][TARGET_COLUMN].values, drug_encoding=DRUG_ENCODING, split_method='no_split')
valid_data = utils.data_process(X_drug=splits['valid'][SMILES_COLUMN].values, y=splits['valid'][TARGET_COLUMN].values, drug_encoding=DRUG_ENCODING, split_method='no_split')
test_data = utils.data_process(X_drug=splits['test'][SMILES_COLUMN].values, y=splits['test'][TARGET_COLUMN].values, drug_encoding=DRUG_ENCODING, split_method='no_split')

config = utils.generate_config(drug_encoding=DRUG_ENCODING, train_epoch=20, LR=0.001, batch_size=128)
model = CompoundPred.model_initialize(**config)
model.train(train_data, valid_data, test_data)
predictions = np.asarray(model.predict(test_data)).ravel()


Drug Property Prediction Mode...
in total: 5170 drugs
encoding drug...
unique drugs: 5138
do not do train/test split on the data for already splitted data
Drug Property Prediction Mode...
in total: 738 drugs
encoding drug...
unique drugs: 737
do not do train/test split on the data for already splitted data
Drug Property Prediction Mode...
in total: 1477 drugs
encoding drug...
unique drugs: 1477
do not do train/test split on the data for already splitted data
Let's use 1 GPU!
--- Data Preparation ---
--- Go for Training ---


DGLError: [04:06:20] /opt/dgl/src/runtime/c_runtime_api.cc:82: Check failed: allow_missing: Device API cuda is not enabled. Please install the cuda version of dgl.
Stack trace:
  [bt] (0) /home/tommaso/miniconda3/envs/tox_prediction/lib/python3.10/site-packages/dgl/libdgl.so(dmlc::LogMessageFatal::~LogMessageFatal()+0x75) [0x7f9a8273e8f5]
  [bt] (1) /home/tommaso/miniconda3/envs/tox_prediction/lib/python3.10/site-packages/dgl/libdgl.so(dgl::runtime::DeviceAPIManager::GetAPI(std::string, bool)+0x202) [0x7f9a82aada92]
  [bt] (2) /home/tommaso/miniconda3/envs/tox_prediction/lib/python3.10/site-packages/dgl/libdgl.so(dgl::runtime::DeviceAPI::Get(DGLContext, bool)+0x1e1) [0x7f9a82aaa071]
  [bt] (3) /home/tommaso/miniconda3/envs/tox_prediction/lib/python3.10/site-packages/dgl/libdgl.so(dgl::runtime::NDArray::Empty(std::vector<long, std::allocator<long> >, DGLDataType, DGLContext)+0x13b) [0x7f9a82ac554b]
  [bt] (4) /home/tommaso/miniconda3/envs/tox_prediction/lib/python3.10/site-packages/dgl/

In [ ]:
y_test = splits['test'][TARGET_COLUMN].to_numpy()
metrics = regression_metrics(y_test, predictions)
model.save_model(str(paths['model']))
save_predictions(paths['predictions'], y_test, predictions)
plot_pred_vs_real(paths['pred_vs_real'], y_test, predictions, 'DeepPurpose GNN: Predicted vs Real')
with paths['metadata'].open('w', encoding='utf-8') as fp:
    json.dump({'run_id': RUN_ID, 'drug_encoding': DRUG_ENCODING, 'metrics': metrics}, fp, indent=2)
metrics
